# PROYECTO PTIA 2026-2
## StudyFlow AI: persistencia (SQLite)

**Integrantes:**

Karol Ximena Rodriguez Reyes
Juan David Moreno D'Aleman

---

### ¿Qué vamos a hacer en este notebook?

`Front (TypeScript)` <-> `API (FastAPI)` <-> `Intérprete -> Validador -> MLP -> Capa de asignación -> Redactor`

`API (FastAPI)` <-> **`Persistencia (SQLite)`**

Hasta ahora todo vivía en memoria: si el estudiante cierra la página, se pierde su plan. Esta capa guarda lo necesario para retomarlo: el plan, el horario que salió del flujo (con las sesiones que el estudiante fijó), la orden que espera confirmación y la decisión que espera su respuesta.

Es una capa **aparte del flujo**. Ninguna capa del flujo la conoce ni la importa: la API lee de la base antes de llamar al flujo y guarda el resultado después. Por eso no cambia nada de lo que ya está probado.

Se prueba aquí antes de pasarla a `studyflow/persistencia.py`. Usa `sqlite3`, que viene con Python, así que no agrega dependencias.

# PASO 0. DEPENDENCIAS

A diferencia de los notebooks anteriores, este se creó **después** de la mudanza a `.py`, así que importa los modelos de `studyflow/` en vez de copiarlos. Necesita el entorno del proyecto (`uv sync`) como kernel.

In [1]:
from datetime import date, datetime, time, timedelta

from studyflow.asignacion import Prediccion, ResultadoAsignacion, Sesion, asignar
from studyflow.interprete import Orden
from studyflow.models import PlanEntrada
from studyflow.ordenes import Decision, PlanVigente, aplicar, crear_plan
from studyflow.validador import Estado, validar

# PASO 1. REGLAS Y DECISIONES

## 1.1 Dónde encaja

Quien usa esta capa es la API:

| Momento | Qué hace la API con la base |
|---|---|
| Primera visita | `nuevo_estudiante` y entrega el identificador al front |
| El estudiante escribe una orden que cambia algo | `leer_decision_pendiente` (si hay una, sus opciones van al contexto del intérprete) -> intérprete -> `guardar_orden_pendiente` -> el front muestra la confirmación |
| El estudiante confirma | `tomar_orden_pendiente` -> `cargar_plan` y `leer_decision_pendiente` -> se aplica la orden (capa de órdenes) -> una de las tres filas siguientes -> redactor |
| El estudiante pulsa "no la hice" en una sesión | `cargar_plan` y `leer_decision_pendiente` -> `marcar_no_hecha` -> una de las tres filas siguientes |
| La orden quedó aplicada | `guardar_plan` con el plan, el horario y las sesiones fijadas nuevos; esa misma transacción borra la decisión pendiente |
| El cambio no cabe en el tiempo disponible | `guardar_decision_pendiente`; el plan guardado no cambia y el front muestra las opciones |
| El estudiante cancela el cambio que no cabía | `descartar_decision_pendiente` |
| El estudiante no confirma | `descartar_orden_pendiente` |
| Una consulta (¿qué tengo mañana?) o vuelve a abrir la página | `cargar_plan` -> `filtrar` y redactor. No pasa por el MLP |

## 1.2 Qué se guarda y qué no

| Se guarda | Por qué |
|---|---|
| El **estudiante**, con un identificador anónimo | la página no tiene login: el front guarda el identificador y lo envía en cada petición |
| El **plan vigente**: actividades, franjas de disponibilidad, duración de sesión, máximo diario y `fecha_inicio` | para replanificar hace falta la entrada completa, no solo el horario |
| El **horario vigente**: sesiones y resumen por actividad (con la prioridad del MLP) | al volver a abrir la página se muestra sin recalcular, y las consultas no gastan MLP ni LLM |
| Las **sesiones fijadas**: una marca en las sesiones del horario | lo que el estudiante movió o fijó no se vuelve a mover en las replanificaciones mientras siga cabiendo en el plan |
| La **orden pendiente** de confirmación | HTTP no recuerda nada entre peticiones: la orden ya interpretada espera aquí a que el estudiante diga que sí |
| La **decisión pendiente**: el plan propuesto con su horario y sus fijadas, las opciones, las horas que faltan y la fecha crítica | cuando un cambio no cabe, el estudiante responde en otra petición y la API necesita recuperar exactamente ese cambio |

| No se guarda | Por qué |
|---|---|
| El texto del chat (mensajes y respuestas del redactor) | no hace falta para planificar y es lo más personal; el texto del horario se puede volver a redactar |
| El resultado del validador | se recalcula en cada cambio |
| Qué sesiones ya se cumplieron | no hace falta marcarlas: una sesión que ya empezó cuenta como hecha por su fecha y hora, y si el estudiante no la hizo, `marcar_no_hecha` la quita del horario |
| Planes y órdenes anteriores | por ahora solo el estado actual (ver 1.5) |

## 1.3 Esquema

| Tabla | Llave | Columnas | Notas |
|---|---|---|---|
| `estudiantes` | `id` | `creado_en` | una fila por estudiante (navegador) |
| `planes` | `estudiante_id` | `fecha_inicio`, `duracion_sesion`, `max_horas_dia`, `actualizado_en` | un plan vigente por estudiante |
| `actividades` | `estudiante_id`, `posicion` | `nombre`, `prioridad`, `dificultad`, `fecha_limite`, `horas_estimadas` | lo que pidió el estudiante |
| `franjas` | `estudiante_id`, `dia`, `inicio` | `fin` | la disponibilidad, una fila por franja |
| `sesiones` | `estudiante_id`, `dia`, `inicio` | `fin`, `actividad`, `fijada` (0 o 1) | salida de la asignación; `fijada = 1` si el estudiante la movió o la fijó |
| `resumen` | `estudiante_id`, `posicion` | `nombre`, `prioridad` (del MLP), `horas_solicitadas`, `horas_asignadas`, `horas_faltantes`, `estado` | salida de la asignación, en orden de prioridad |
| `orden_pendiente` | `estudiante_id` | `orden` (JSON), `creada_en` | como mucho una por estudiante |
| `decision_pendiente` | `estudiante_id` | `decision` (JSON), `creada_en` | como mucho una por estudiante |

`planes` depende de `estudiantes`, y las otras cuatro tablas del plan dependen de `planes` con `ON DELETE CASCADE`: al borrar un plan se borra todo lo suyo. `orden_pendiente` y `decision_pendiente` dependen de `estudiantes`.

## 1.4 Decisiones

* **`sqlite3` de la librería estándar, sin ORM.** Son ocho tablas pequeñas con consultas simples.
* **Fechas y horas como texto ISO** (`2026-10-26`, `18:00:00`). SQLite no tiene tipo fecha, y el texto ISO se ordena bien.
* **Identificador anónimo:** `uuid4` en hexadecimal (32 caracteres), imposible de adivinar. Como es lo único que separa a un estudiante de otro, funciona como una contraseña.
* **Las actividades se guardan por posición, no por nombre.** Las predicciones del MLP se alinean por posición, y el validador permite nombres repetidos (solo advierte, R6).
* **Los días sin franjas no se guardan.** Un día bloqueado tiene capacidad 0, igual que un día que no aparece. El apartado 3.2 comprueba que la validación y la asignación dan lo mismo.
* **Guardar es reemplazar, en una sola transacción.** Se borra el plan anterior (la cascada se lleva sus actividades, franjas, sesiones y resumen) y se inserta el nuevo. Si algo falla a mitad de camino, queda el anterior completo.
* **El plan y el horario se guardan siempre juntos, con sus fijadas,** y solo cuando el plan no es inviable. Si un cambio deja el plan inviable, el estudiante conserva el que tenía.
* **Una fijada es una sesión del horario con una marca, no una tabla aparte.** Así nunca puede haber una fijada que no esté en el horario, y `guardar_plan` rechaza (`ValueError`) la que no lo esté. `cargar_plan` las devuelve en una lista aparte, que es lo que la capa de órdenes recibe en `PlanVigente.fijadas`.
* **Al cargar, todo vuelve a pasar por Pydantic.** Si algo inválido llegara a la base, falla al cargar y no llega al flujo. Los `CHECK` del esquema son una segunda barrera para los valores básicos.
* **Una sola orden pendiente por estudiante, que vence a los 30 minutos.** Si escribe otra orden antes de confirmar, la nueva reemplaza a la anterior. El vencimiento evita aplicar fechas viejas: el intérprete ya convirtió mañana en una fecha, y confirmarla días después sería otro día.
* **Una sola decisión pendiente por estudiante, que también vence a los 30 minutos.** Se guarda completa como JSON (la `Decision` de la capa de órdenes), igual que la orden. Guardar otra la reemplaza y reinicia el vencimiento.
* **La decisión se lee sin borrarla.** Mientras decide, el estudiante puede preguntar algo, pedir una orden que se rechaza o pedir la hora de un recorte, y la decisión debe seguir ahí. Se borra solo cuando se resuelve (`guardar_plan`) o se cancela (`descartar_decision_pendiente`).
* **La decisión coexiste con la orden pendiente.** Responder a una decisión es una orden más (`responder_decision`) y también espera confirmación, así que pueden estar las dos a la vez.
* **`guardar_plan` borra la decisión pendiente en su misma transacción.** La decisión habla del plan que había: con un plan nuevo ya no tiene sentido, y si el guardado falla, la decisión sigue donde estaba.

## 1.5 Lo que todavía no se guarda

**Historial.** Hoy se guarda solo el estado actual. Un historial de órdenes permitiría deshacer cambios y medir en el informe cómo se usa el chat, a cambio de guardar más datos personales. Sería una tabla nueva y no cambia nada de lo demás.

# PASO 2. ESQUEMA Y CONEXIÓN (`persistencia.py`, primera parte)

`ESQUEMA` crea las ocho tablas del apartado 1.3. `conectar` abre la base (la crea si no existe), activa las llaves foráneas, que SQLite trae apagadas, y crea las tablas que falten, así que se puede llamar cada vez sin borrar nada. Con `":memory:"` la base vive solo en memoria, que es lo que usan las pruebas.

In [2]:
"""Persistencia con SQLite: esquema y conexión. Ver StudyFlow_Persistencia.ipynb, paso 2."""
import sqlite3
import uuid
from datetime import date, datetime, timedelta

from pydantic import BaseModel

ESQUEMA = """
CREATE TABLE IF NOT EXISTS estudiantes (
    id        TEXT PRIMARY KEY,
    creado_en TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS planes (
    estudiante_id   TEXT PRIMARY KEY REFERENCES estudiantes(id) ON DELETE CASCADE,
    fecha_inicio    TEXT NOT NULL,
    duracion_sesion INTEGER NOT NULL CHECK (duracion_sesion BETWEEN 30 AND 240 AND duracion_sesion % 30 = 0),
    max_horas_dia   REAL CHECK (max_horas_dia IS NULL OR (max_horas_dia > 0 AND max_horas_dia <= 8)),
    actualizado_en  TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS actividades (
    estudiante_id   TEXT NOT NULL REFERENCES planes(estudiante_id) ON DELETE CASCADE,
    posicion        INTEGER NOT NULL,
    nombre          TEXT NOT NULL,
    prioridad       TEXT NOT NULL CHECK (prioridad IN ('baja', 'media', 'alta')),
    dificultad      TEXT NOT NULL CHECK (dificultad IN ('baja', 'media', 'alta')),
    fecha_limite    TEXT NOT NULL,
    horas_estimadas REAL NOT NULL CHECK (horas_estimadas > 0 AND horas_estimadas <= 100),
    PRIMARY KEY (estudiante_id, posicion)
);

CREATE TABLE IF NOT EXISTS franjas (
    estudiante_id TEXT NOT NULL REFERENCES planes(estudiante_id) ON DELETE CASCADE,
    dia           TEXT NOT NULL,
    inicio        TEXT NOT NULL,
    fin           TEXT NOT NULL CHECK (fin > inicio),
    PRIMARY KEY (estudiante_id, dia, inicio)
);

CREATE TABLE IF NOT EXISTS sesiones (
    estudiante_id TEXT NOT NULL REFERENCES planes(estudiante_id) ON DELETE CASCADE,
    dia           TEXT NOT NULL,
    inicio        TEXT NOT NULL,
    fin           TEXT NOT NULL CHECK (fin > inicio),
    actividad     TEXT NOT NULL,
    fijada        INTEGER NOT NULL DEFAULT 0 CHECK (fijada IN (0, 1)),
    PRIMARY KEY (estudiante_id, dia, inicio)
);

CREATE TABLE IF NOT EXISTS resumen (
    estudiante_id     TEXT NOT NULL REFERENCES planes(estudiante_id) ON DELETE CASCADE,
    posicion          INTEGER NOT NULL,
    nombre            TEXT NOT NULL,
    prioridad         REAL NOT NULL CHECK (prioridad BETWEEN 0 AND 1),
    horas_solicitadas REAL NOT NULL,
    horas_asignadas   REAL NOT NULL,
    horas_faltantes   REAL NOT NULL,
    estado            TEXT NOT NULL CHECK (estado IN ('asignada', 'parcial', 'no_asignada')),
    PRIMARY KEY (estudiante_id, posicion)
);

CREATE TABLE IF NOT EXISTS orden_pendiente (
    estudiante_id TEXT PRIMARY KEY REFERENCES estudiantes(id) ON DELETE CASCADE,
    orden         TEXT NOT NULL,
    creada_en     TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS decision_pendiente (
    estudiante_id TEXT PRIMARY KEY REFERENCES estudiantes(id) ON DELETE CASCADE,
    decision      TEXT NOT NULL,
    creada_en     TEXT NOT NULL
);
"""


def conectar(ruta: str = "studyflow.db") -> sqlite3.Connection:
    """Abre la base (la crea si no existe), activa las llaves foráneas y crea las tablas que falten."""
    con = sqlite3.connect(ruta)
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(ESQUEMA)
    return con


def _ahora() -> str:
    """Fecha y hora actuales en texto ISO, sin microsegundos."""
    return datetime.now().isoformat(timespec="seconds")

## 2.1 Pruebas del esquema

Cada prueba es una función `test_...` con su propia base nueva en memoria. `correr` las ejecuta y muestra cuáles pasaron; al migrar, las funciones pasan tal cual a pytest.

In [3]:
import tempfile
from pathlib import Path


def correr(*pruebas):
    """Ejecuta cada prueba y muestra su nombre si pasó."""
    for prueba in pruebas:
        prueba()
        print("ok", prueba.__name__)


def base() -> sqlite3.Connection:
    """Base nueva en memoria para una prueba."""
    return conectar(":memory:")


def contar(con: sqlite3.Connection, tabla: str) -> int:
    """Número de filas de una tabla."""
    return con.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]


def falla_integridad(fn, fragmento: str):
    """Comprueba que la base rechace fn con un error que contiene el fragmento."""
    try:
        fn()
    except sqlite3.IntegrityError as e:
        assert fragmento in str(e), str(e)
        return
    raise AssertionError(f"la base debía rechazarlo ({fragmento})")


def test_crea_todas_las_tablas():
    """conectar crea las ocho tablas del esquema."""
    tablas = {fila["name"] for fila in base().execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    assert tablas == {"estudiantes", "planes", "actividades", "franjas", "sesiones", "resumen", "orden_pendiente", "decision_pendiente"}


def test_reconectar_no_borra_nada():
    """Abrir otra vez el mismo archivo conserva los datos, porque las tablas solo se crean si faltan."""
    ruta = str(Path(tempfile.mkdtemp()) / "prueba.db")
    con = conectar(ruta)
    con.execute("INSERT INTO estudiantes VALUES ('e1', '2026-10-25T10:00:00')")
    con.commit()
    con.close()
    assert conectar(ruta).execute("SELECT id FROM estudiantes").fetchone()["id"] == "e1"


def test_llaves_foraneas_activas():
    """La base no acepta un plan de un estudiante que no existe."""
    con = base()
    falla_integridad(lambda: con.execute("INSERT INTO planes VALUES ('nadie', '2026-10-25', 60, NULL, '2026-10-25T10:00:00')"), "FOREIGN KEY")


def test_checks_rechazan_valores_invalidos():
    """Los CHECK del esquema frenan una duración, un nivel, una franja y una marca de fijada inválidos."""
    con = base()
    con.execute("INSERT INTO estudiantes VALUES ('e1', '2026-10-25T10:00:00')")
    falla_integridad(lambda: con.execute("INSERT INTO planes VALUES ('e1', '2026-10-25', 45, NULL, '2026-10-25T10:00:00')"), "CHECK")
    con.execute("INSERT INTO planes VALUES ('e1', '2026-10-25', 60, NULL, '2026-10-25T10:00:00')")
    falla_integridad(lambda: con.execute("INSERT INTO actividades VALUES ('e1', 0, 'Taller', 'urgente', 'media', '2026-10-30', 3)"), "CHECK")
    falla_integridad(lambda: con.execute("INSERT INTO franjas VALUES ('e1', '2026-10-26', '21:00:00', '18:00:00')"), "CHECK")
    falla_integridad(lambda: con.execute("INSERT INTO sesiones VALUES ('e1', '2026-10-26', '18:00:00', '19:00:00', 'Taller', 2)"), "CHECK")


def test_borrar_el_plan_borra_todo_lo_suyo():
    """Con ON DELETE CASCADE, al borrar un plan se van sus actividades y franjas."""
    con = base()
    con.execute("INSERT INTO estudiantes VALUES ('e1', '2026-10-25T10:00:00')")
    con.execute("INSERT INTO planes VALUES ('e1', '2026-10-25', 60, NULL, '2026-10-25T10:00:00')")
    con.execute("INSERT INTO actividades VALUES ('e1', 0, 'Taller', 'media', 'media', '2026-10-30', 3)")
    con.execute("INSERT INTO franjas VALUES ('e1', '2026-10-26', '18:00:00', '21:00:00')")
    con.execute("DELETE FROM planes WHERE estudiante_id = 'e1'")
    assert contar(con, "actividades") == contar(con, "franjas") == 0 and contar(con, "estudiantes") == 1


def test_borrar_al_estudiante_borra_lo_que_espera():
    """Al borrar un estudiante se van también su orden y su decisión pendientes."""
    con = base()
    con.execute("INSERT INTO estudiantes VALUES ('e1', '2026-10-25T10:00:00')")
    con.execute("INSERT INTO orden_pendiente VALUES ('e1', '{}', '2026-10-25T10:00:00')")
    con.execute("INSERT INTO decision_pendiente VALUES ('e1', '{}', '2026-10-25T10:00:00')")
    con.execute("DELETE FROM estudiantes WHERE id = 'e1'")
    assert contar(con, "orden_pendiente") == contar(con, "decision_pendiente") == 0


correr(test_crea_todas_las_tablas, test_reconectar_no_borra_nada, test_llaves_foraneas_activas,
       test_checks_rechazan_valores_invalidos, test_borrar_el_plan_borra_todo_lo_suyo, test_borrar_al_estudiante_borra_lo_que_espera)

ok test_crea_todas_las_tablas
ok test_reconectar_no_borra_nada
ok test_llaves_foraneas_activas
ok test_checks_rechazan_valores_invalidos
ok test_borrar_el_plan_borra_todo_lo_suyo
ok test_borrar_al_estudiante_borra_lo_que_espera


# PASO 3. ESTUDIANTES, PLAN Y HORARIO (`persistencia.py`, segunda parte)

| Función | Qué hace |
|---|---|
| `nuevo_estudiante` | registra un estudiante anónimo y devuelve su identificador |
| `existe_estudiante` | revisa un identificador que llega del front |
| `guardar_plan` | reemplaza el plan, el horario y las sesiones fijadas en una sola transacción, y borra la decisión pendiente |
| `cargar_plan` | devuelve un `PlanGuardado` (plan, `fecha_inicio`, horario, fijadas y fecha de guardado), o `None` si el estudiante aún no tiene plan |

`with con:` abre la transacción: si todo sale bien la confirma, y si algo falla la deshace completa.

Las fijadas son sesiones del horario: `guardar_plan` marca con `fijada = 1` las que reciba y `cargar_plan` las devuelve aparte (el horario sigue trayéndolas todas). Con `PlanVigente(plan=g.plan, horario=g.horario, fijadas=g.fijadas)` el resultado de `cargar_plan` es justo lo que la capa de órdenes necesita.

In [4]:
"""Persistencia con SQLite: estudiantes, plan y horario. Ver StudyFlow_Persistencia.ipynb, paso 3."""


class PlanGuardado(BaseModel):
    """Lo que se recupera de un estudiante: su plan, la fecha desde la que se planificó, el horario vigente y las sesiones fijadas."""

    plan: PlanEntrada
    fecha_inicio: date
    horario: ResultadoAsignacion
    fijadas: list[Sesion]
    actualizado_en: datetime


def nuevo_estudiante(con: sqlite3.Connection) -> str:
    """Registra un estudiante anónimo y devuelve su identificador."""
    estudiante_id = uuid.uuid4().hex
    with con:
        con.execute("INSERT INTO estudiantes VALUES (?, ?)", (estudiante_id, _ahora()))
    return estudiante_id


def existe_estudiante(con: sqlite3.Connection, estudiante_id: str) -> bool:
    """Indica si el identificador pertenece a un estudiante registrado."""
    return con.execute("SELECT 1 FROM estudiantes WHERE id = ?", (estudiante_id,)).fetchone() is not None


def guardar_plan(con: sqlite3.Connection, estudiante_id: str, plan: PlanEntrada, fecha_inicio: date,
                 horario: ResultadoAsignacion, fijadas: list[Sesion] | None = None) -> None:
    """Reemplaza el plan, el horario y las fijadas del estudiante en una sola transacción: se guarda todo o nada."""
    e, fijadas = estudiante_id, fijadas or []
    if any(s not in horario.sesiones for s in fijadas):
        raise ValueError("Una sesión fijada no está en el horario.")
    with con:
        con.execute("DELETE FROM planes WHERE estudiante_id = ?", (e,))
        con.execute("DELETE FROM decision_pendiente WHERE estudiante_id = ?", (e,))
        con.execute("INSERT INTO planes VALUES (?, ?, ?, ?, ?)",
                    (e, fecha_inicio.isoformat(), int(plan.duracion_sesion), plan.max_horas_dia, _ahora()))
        con.executemany("INSERT INTO actividades VALUES (?, ?, ?, ?, ?, ?, ?)",
                        [(e, i, a.nombre, a.prioridad.value, a.dificultad.value, a.fecha_limite.isoformat(), a.horas_estimadas)
                         for i, a in enumerate(plan.actividades)])
        con.executemany("INSERT INTO franjas VALUES (?, ?, ?, ?)",
                        [(e, d.dia.isoformat(), fr.inicio.isoformat(), fr.fin.isoformat()) for d in plan.disponibilidad for fr in d.franjas])
        con.executemany("INSERT INTO sesiones VALUES (?, ?, ?, ?, ?, ?)",
                        [(e, s.dia.isoformat(), s.inicio.isoformat(), s.fin.isoformat(), s.actividad, int(s in fijadas))
                         for s in horario.sesiones])
        con.executemany("INSERT INTO resumen VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                        [(e, i, a.nombre, a.prioridad, a.horas_solicitadas, a.horas_asignadas, a.horas_faltantes, a.estado.value)
                         for i, a in enumerate(horario.actividades)])


def _filas(con: sqlite3.Connection, consulta: str, estudiante_id: str) -> list[dict]:
    """Ejecuta una consulta filtrada por estudiante y devuelve las filas como diccionarios."""
    return [dict(fila) for fila in con.execute(consulta, (estudiante_id,))]


def cargar_plan(con: sqlite3.Connection, estudiante_id: str) -> PlanGuardado | None:
    """Recupera el plan, el horario y las fijadas vigentes del estudiante, o None si todavía no tiene plan."""
    filas = _filas(con, "SELECT * FROM planes WHERE estudiante_id = ?", estudiante_id)
    if not filas:
        return None
    p = filas[0]
    dias_con_franjas = {}
    for fr in _filas(con, "SELECT dia, inicio, fin FROM franjas WHERE estudiante_id = ? ORDER BY dia, inicio", estudiante_id):
        dias_con_franjas.setdefault(fr.pop("dia"), []).append(fr)
    plan = {"actividades": _filas(con, "SELECT nombre, prioridad, dificultad, fecha_limite, horas_estimadas FROM actividades "
                                       "WHERE estudiante_id = ? ORDER BY posicion", estudiante_id),
            "disponibilidad": [{"dia": d, "franjas": frs} for d, frs in dias_con_franjas.items()],
            "duracion_sesion": p["duracion_sesion"], "max_horas_dia": p["max_horas_dia"]}
    sesiones = _filas(con, "SELECT actividad, dia, inicio, fin, fijada FROM sesiones WHERE estudiante_id = ? ORDER BY dia, inicio",
                      estudiante_id)
    horario = {"sesiones": [{k: v for k, v in s.items() if k != "fijada"} for s in sesiones],
               "actividades": _filas(con, "SELECT nombre, prioridad, horas_solicitadas, horas_asignadas, horas_faltantes, estado "
                                          "FROM resumen WHERE estudiante_id = ? ORDER BY posicion", estudiante_id)}
    fijadas = [{k: v for k, v in s.items() if k != "fijada"} for s in sesiones if s["fijada"]]
    return PlanGuardado(plan=plan, fecha_inicio=p["fecha_inicio"], horario=horario, fijadas=fijadas, actualizado_en=p["actualizado_en"])

## 3.1 Datos de prueba

Los mismos ayudantes de las pruebas de la asignación (`INICIO` = domingo 25 de octubre de 2026, `act`, `dias`, `plan`), que ya están en `tests/conftest.py`. El plan de ejemplo es el del paso 4 de `StudyFlow_CapaAsignacion.ipynb`, con las predicciones escritas a mano: esta capa no necesita el MLP.

`normalizar` deja un plan como vuelve de la base: sin días vacíos y con días y franjas en orden.

In [5]:
INICIO = date(2026, 10, 25)  # domingo
PREDS_EJEMPLO = [Prediccion(prioridad=0.87, perfil=[0.45, 0.35, 0.20]), Prediccion(prioridad=0.55, perfil=[0.30, 0.40, 0.30])]
UNIF = [1 / 3] * 3


def f(n: int, inicio: date = INICIO) -> str:
    """Fecha n días después del inicio, en texto ISO."""
    return (inicio + timedelta(days=n)).isoformat()


def franjas(*pares) -> list[dict]:
    """Franjas a partir de pares (inicio, fin)."""
    return [{"inicio": i, "fin": fin} for i, fin in pares]


def act(nombre, n_limite, horas, prioridad="media", dificultad="media") -> dict:
    """Actividad con fecha límite n días después del inicio."""
    return {"nombre": nombre, "prioridad": prioridad, "dificultad": dificultad, "fecha_limite": f(n_limite), "horas_estimadas": horas}


def dias(desde, hasta, pares=(("18:00", "21:00"),)) -> list[dict]:
    """Disponibilidad igual para los días desde..hasta-1."""
    return [{"dia": f(n), "franjas": franjas(*pares)} for n in range(desde, hasta)]


def plan(actividades, disponibilidad, sesion=60, max_horas_dia=None) -> dict:
    """Plan de entrada como diccionario."""
    return {"actividades": actividades, "disponibilidad": disponibilidad, "duracion_sesion": sesion, "max_horas_dia": max_horas_dia}


def ejemplo() -> tuple[PlanEntrada, ResultadoAsignacion]:
    """Plan del paso 4 de la asignación (Parcial de IA y Taller) con su horario."""
    p = PlanEntrada(**plan([act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)], dias(0, 10)))
    return p, asignar(p, PREDS_EJEMPLO, INICIO)


def normalizar(p: PlanEntrada) -> PlanEntrada:
    """El plan como vuelve de la base: sin días vacíos y con días y franjas en orden."""
    disp = [d.model_copy(update={"franjas": sorted(d.franjas, key=lambda fr: fr.inicio)}) for d in p.disponibilidad if d.franjas]
    return p.model_copy(update={"disponibilidad": sorted(disp, key=lambda d: d.dia)})

## 3.2 Pruebas de guardar y cargar

In [6]:
def test_estudiante_nuevo_sin_plan():
    """Un estudiante recién creado existe, tiene un identificador de 32 caracteres y aún no tiene plan."""
    con = base()
    e1, e2 = nuevo_estudiante(con), nuevo_estudiante(con)
    assert e1 != e2 and len(e1) == 32
    assert existe_estudiante(con, e1) and not existe_estudiante(con, "no-existe")
    assert cargar_plan(con, e1) is None


def test_guardar_y_cargar_devuelve_lo_mismo():
    """Lo que se carga es igual a lo que se guardó: plan, fecha de inicio y horario."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r)
    g = cargar_plan(con, e)
    assert g.plan == p and g.fecha_inicio == INICIO and g.horario == r


def test_guardar_otra_vez_reemplaza():
    """Guardar un plan nuevo reemplaza al anterior y no deja filas viejas."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r)
    p2 = PlanEntrada(**plan([act("Lectura", 4, 2, "baja", "baja")], dias(1, 5), sesion=30))
    r2 = asignar(p2, [Prediccion(prioridad=0.3, perfil=UNIF)], INICIO + timedelta(days=1))
    guardar_plan(con, e, p2, INICIO + timedelta(days=1), r2)
    g = cargar_plan(con, e)
    assert g.plan == p2 and g.horario == r2 and g.fecha_inicio == INICIO + timedelta(days=1)
    assert [contar(con, t) for t in ("planes", "actividades", "franjas", "resumen", "sesiones")] == [1, 1, 4, 1, len(r2.sesiones)]


def test_si_algo_falla_queda_el_plan_anterior():
    """Si el guardado falla a mitad de camino, la transacción se deshace y el plan anterior sigue completo."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r)
    roto = r.model_copy(update={"sesiones": r.sesiones + [r.sesiones[0]]})  # sesión repetida: viola la llave primaria
    falla_integridad(lambda: guardar_plan(con, e, p, INICIO + timedelta(days=1), roto), "UNIQUE")
    g = cargar_plan(con, e)
    assert g.plan == p and g.horario == r and g.fecha_inicio == INICIO


def test_estudiante_inexistente_no_guarda_nada():
    """No se puede guardar un plan para un identificador que no está registrado."""
    con = base()
    p, r = ejemplo()
    falla_integridad(lambda: guardar_plan(con, "no-existe", p, INICIO, r), "FOREIGN KEY")
    assert contar(con, "planes") == contar(con, "actividades") == 0


def test_estudiantes_no_se_mezclan():
    """Cada estudiante recupera solo su propio plan."""
    con = base()
    e1, e2 = nuevo_estudiante(con), nuevo_estudiante(con)
    p, r = ejemplo()
    p2 = PlanEntrada(**plan([act("Lectura", 4, 2)], dias(0, 5)))
    r2 = asignar(p2, [Prediccion(prioridad=0.5, perfil=UNIF)], INICIO)
    guardar_plan(con, e1, p, INICIO, r)
    guardar_plan(con, e2, p2, INICIO, r2)
    assert cargar_plan(con, e1).plan == p and cargar_plan(con, e2).plan == p2


def test_nombres_repetidos_conservan_su_posicion():
    """Dos actividades con el mismo nombre vuelven en el mismo orden, para que las predicciones sigan alineadas."""
    con = base()
    e = nuevo_estudiante(con)
    p = PlanEntrada(**plan([act("Lectura", 6, 3, "alta"), act("Lectura", 8, 1, "baja")], dias(0, 10)))
    r = asignar(p, [Prediccion(prioridad=0.8, perfil=UNIF), Prediccion(prioridad=0.2, perfil=UNIF)], INICIO)
    guardar_plan(con, e, p, INICIO, r)
    assert [(a.horas_estimadas, a.prioridad.value) for a in cargar_plan(con, e).plan.actividades] == [(3, "alta"), (1, "baja")]


def test_dias_sin_franjas_no_cambian_el_resultado():
    """Un día bloqueado (sin franjas) no se guarda, y eso no cambia ni la validación ni la asignación."""
    con = base()
    e = nuevo_estudiante(con)
    disp = dias(0, 10)
    disp[2]["franjas"] = []  # el estudiante bloqueó ese día
    p = PlanEntrada(**plan([act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)], disp))
    r = asignar(p, PREDS_EJEMPLO, INICIO)
    guardar_plan(con, e, p, INICIO, r)
    cargado = cargar_plan(con, e).plan
    assert len(cargado.disponibilidad) == 9 and cargado == normalizar(p)
    assert validar(cargado, INICIO) == validar(p, INICIO)
    assert asignar(cargado, PREDS_EJEMPLO, INICIO) == r


def test_fijadas_se_guardan_y_vuelven_marcadas():
    """Las sesiones fijadas vuelven en su propia lista, y el horario sigue trayendo todas las sesiones."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    fijadas = [r.sesiones[2], r.sesiones[7]]
    guardar_plan(con, e, p, INICIO, r, fijadas)
    g = cargar_plan(con, e)
    assert g.fijadas == fijadas and g.horario == r
    assert con.execute("SELECT COUNT(*) FROM sesiones WHERE fijada = 1").fetchone()[0] == 2


def test_sin_fijadas_ninguna_queda_marcada():
    """Si no se pasan fijadas, ninguna sesión queda marcada."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r)
    assert cargar_plan(con, e).fijadas == []


def test_guardar_otra_vez_reemplaza_las_fijadas():
    """Un plan guardado sin fijadas no hereda las del anterior."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r, [r.sesiones[0]])
    guardar_plan(con, e, p, INICIO, r)
    assert cargar_plan(con, e).fijadas == []


def test_fijada_que_no_esta_en_el_horario_se_rechaza():
    """Una fijada que no es del horario se rechaza sin tocar el plan anterior."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_plan(con, e, p, INICIO, r, [r.sesiones[0]])
    intrusa = r.sesiones[1].model_copy(update={"actividad": "Otra"})
    try:
        guardar_plan(con, e, p, INICIO + timedelta(days=1), r, [intrusa])
    except ValueError as err:
        assert "fijada" in str(err)
    else:
        raise AssertionError("debía rechazar una fijada que no está en el horario")
    g = cargar_plan(con, e)
    assert g.fecha_inicio == INICIO and g.fijadas == [r.sesiones[0]]


correr(test_estudiante_nuevo_sin_plan, test_guardar_y_cargar_devuelve_lo_mismo, test_guardar_otra_vez_reemplaza,
       test_si_algo_falla_queda_el_plan_anterior, test_estudiante_inexistente_no_guarda_nada, test_estudiantes_no_se_mezclan,
       test_nombres_repetidos_conservan_su_posicion, test_dias_sin_franjas_no_cambian_el_resultado,
       test_fijadas_se_guardan_y_vuelven_marcadas, test_sin_fijadas_ninguna_queda_marcada,
       test_guardar_otra_vez_reemplaza_las_fijadas, test_fijada_que_no_esta_en_el_horario_se_rechaza)

ok test_estudiante_nuevo_sin_plan
ok test_guardar_y_cargar_devuelve_lo_mismo
ok test_guardar_otra_vez_reemplaza
ok test_si_algo_falla_queda_el_plan_anterior
ok test_estudiante_inexistente_no_guarda_nada
ok test_estudiantes_no_se_mezclan
ok test_nombres_repetidos_conservan_su_posicion
ok test_dias_sin_franjas_no_cambian_el_resultado
ok test_fijadas_se_guardan_y_vuelven_marcadas
ok test_sin_fijadas_ninguna_queda_marcada
ok test_guardar_otra_vez_reemplaza_las_fijadas
ok test_fijada_que_no_esta_en_el_horario_se_rechaza


# PASO 4. ORDEN Y DECISIÓN PENDIENTES (`persistencia.py`, tercera parte)

Dos cosas esperan en la base entre una petición y la siguiente.

**La orden pendiente.** Toda orden que cambia algo se confirma antes de aplicarse. Entre la petición que la interpreta y la que la confirma, la orden (ya validada y con las fechas resueltas) espera en `orden_pendiente`.

**La decisión pendiente.** Cuando un cambio no cabe en el tiempo disponible, la capa de órdenes devuelve una `Decision`: el plan propuesto con el cambio ya aplicado, las opciones del estudiante (abrir tiempo, recortar y, si el cambio era un deseo y no algo que ya ocurrió, cancelar), las horas que faltan y la fecha crítica. El estudiante responde en otra petición, así que la decisión espera en `decision_pendiente`.

| Función | Cuándo se usa |
|---|---|
| `guardar_orden_pendiente` | el intérprete devolvió una orden con `requiere_confirmacion` |
| `tomar_orden_pendiente` | el estudiante confirma: la saca de la base para aplicarla; `None` si no había o ya venció |
| `descartar_orden_pendiente` | el estudiante no confirma |
| `guardar_decision_pendiente` | la capa de órdenes devolvió una decisión (o una pregunta que la mantiene abierta) |
| `leer_decision_pendiente` | hay que saber si el estudiante tiene una decisión abierta y qué opciones tiene; `None` si no hay o ya venció |
| `descartar_decision_pendiente` | el estudiante cancela el cambio que no cabía |

Las dos se guardan como JSON (`model_dump_json`) en una sola columna, porque cada orden usa campos distintos y la decisión lleva un plan completo. Al leerlas, Pydantic las reconstruye y las valida otra vez. Ambas vencen a los 30 minutos.

Se diferencian en tres cosas:

* La orden se **toma**: se saca de la base al leerla. La decisión se **lee sin borrarla**, porque el estudiante puede escribir otras cosas antes de resolverla.
* **Coexisten.** Responder a una decisión es una orden (`responder_decision`) que también espera confirmación.
* La decisión la borra **`guardar_plan` en su misma transacción**: habla del plan que había, y con un plan nuevo ya no tiene sentido.

In [7]:
"""Persistencia con SQLite: orden y decisión pendientes. Ver StudyFlow_Persistencia.ipynb, paso 4."""

VIGENCIA_ORDEN = timedelta(minutes=30)
VIGENCIA_DECISION = timedelta(minutes=30)


def guardar_orden_pendiente(con: sqlite3.Connection, estudiante_id: str, orden: Orden) -> None:
    """Guarda la orden que espera confirmación; si ya había otra, la reemplaza."""
    with con:
        con.execute("INSERT OR REPLACE INTO orden_pendiente VALUES (?, ?, ?)", (estudiante_id, orden.model_dump_json(), _ahora()))


def tomar_orden_pendiente(con: sqlite3.Connection, estudiante_id: str) -> Orden | None:
    """Saca la orden pendiente para aplicarla; devuelve None si no había o si ya venció."""
    with con:
        fila = con.execute("SELECT orden, creada_en FROM orden_pendiente WHERE estudiante_id = ?", (estudiante_id,)).fetchone()
        con.execute("DELETE FROM orden_pendiente WHERE estudiante_id = ?", (estudiante_id,))
    if fila is None or datetime.now() - datetime.fromisoformat(fila["creada_en"]) > VIGENCIA_ORDEN:
        return None
    return Orden.model_validate_json(fila["orden"])


def descartar_orden_pendiente(con: sqlite3.Connection, estudiante_id: str) -> None:
    """Borra la orden pendiente porque el estudiante no la confirmó."""
    with con:
        con.execute("DELETE FROM orden_pendiente WHERE estudiante_id = ?", (estudiante_id,))


def guardar_decision_pendiente(con: sqlite3.Connection, estudiante_id: str, decision: Decision) -> None:
    """Guarda la decisión que espera la respuesta del estudiante; si ya había otra, la reemplaza y reinicia el vencimiento."""
    with con:
        con.execute("INSERT OR REPLACE INTO decision_pendiente VALUES (?, ?, ?)", (estudiante_id, decision.model_dump_json(), _ahora()))


def leer_decision_pendiente(con: sqlite3.Connection, estudiante_id: str) -> Decision | None:
    """Devuelve la decisión pendiente sin borrarla; None si no hay o si ya venció."""
    fila = con.execute("SELECT decision, creada_en FROM decision_pendiente WHERE estudiante_id = ?", (estudiante_id,)).fetchone()
    if fila is None or datetime.now() - datetime.fromisoformat(fila["creada_en"]) > VIGENCIA_DECISION:
        return None
    return Decision.model_validate_json(fila["decision"])


def descartar_decision_pendiente(con: sqlite3.Connection, estudiante_id: str) -> None:
    """Borra la decisión pendiente porque el estudiante canceló el cambio."""
    with con:
        con.execute("DELETE FROM decision_pendiente WHERE estudiante_id = ?", (estudiante_id,))


## 4.1 Pruebas de la orden pendiente

In [8]:
ORDEN_CREAR = Orden(accion="crear_plan", actividades=[act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)])
ORDEN_MOVER = Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=f(1), dia_destino=f(6), hora_inicio="09:00")


def test_sin_orden_pendiente():
    """Si no hay orden pendiente, tomar devuelve None y descartar no falla."""
    con = base()
    e = nuevo_estudiante(con)
    assert tomar_orden_pendiente(con, e) is None
    descartar_orden_pendiente(con, e)


def test_tomar_devuelve_la_orden_y_la_saca():
    """La orden vuelve igual a como se guardó (con sus actividades) y no se puede tomar dos veces."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_orden_pendiente(con, e, ORDEN_CREAR)
    assert tomar_orden_pendiente(con, e) == ORDEN_CREAR
    assert tomar_orden_pendiente(con, e) is None


def test_orden_nueva_reemplaza_a_la_anterior():
    """Si el estudiante escribe otra orden antes de confirmar, la que espera es la última."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_orden_pendiente(con, e, ORDEN_CREAR)
    guardar_orden_pendiente(con, e, ORDEN_MOVER)
    assert contar(con, "orden_pendiente") == 1
    assert tomar_orden_pendiente(con, e) == ORDEN_MOVER


def test_descartar_borra_la_orden():
    """Cuando el estudiante no confirma, la orden se borra y ya no se puede aplicar."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_orden_pendiente(con, e, ORDEN_MOVER)
    descartar_orden_pendiente(con, e)
    assert tomar_orden_pendiente(con, e) is None


def test_orden_vencida_no_se_aplica():
    """Una orden de hace más de 30 minutos no se aplica (sus fechas pudieron quedar viejas) y se borra."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_orden_pendiente(con, e, ORDEN_MOVER)
    hace_31_min = (datetime.now() - timedelta(minutes=31)).isoformat(timespec="seconds")
    with con:
        con.execute("UPDATE orden_pendiente SET creada_en = ?", (hace_31_min,))
    assert tomar_orden_pendiente(con, e) is None and contar(con, "orden_pendiente") == 0


def test_orden_de_otro_estudiante_no_se_ve():
    """La orden pendiente de un estudiante no la puede tomar ni descartar otro."""
    con = base()
    e1, e2 = nuevo_estudiante(con), nuevo_estudiante(con)
    guardar_orden_pendiente(con, e1, ORDEN_MOVER)
    descartar_orden_pendiente(con, e2)
    assert tomar_orden_pendiente(con, e2) is None
    assert tomar_orden_pendiente(con, e1) == ORDEN_MOVER


correr(test_sin_orden_pendiente, test_tomar_devuelve_la_orden_y_la_saca, test_orden_nueva_reemplaza_a_la_anterior,
       test_descartar_borra_la_orden, test_orden_vencida_no_se_aplica, test_orden_de_otro_estudiante_no_se_ve)

ok test_sin_orden_pendiente
ok test_tomar_devuelve_la_orden_y_la_saca
ok test_orden_nueva_reemplaza_a_la_anterior
ok test_descartar_borra_la_orden
ok test_orden_vencida_no_se_aplica
ok test_orden_de_otro_estudiante_no_se_ve


## 4.2 Pruebas de la decisión pendiente

`decision_ejemplo` arma una `Decision` real con el plan del ejemplo: es la forma que tendrá la que devuelva la capa de órdenes.

In [9]:
def decision_ejemplo(opciones=("abrir_tiempo", "recortar", "cancelar")) -> Decision:
    """Un cambio que no cabe: el plan propuesto (con una sesión fijada), las opciones, las horas que faltan y la fecha crítica."""
    p, r = ejemplo()
    return Decision(propuesto=PlanVigente(plan=p, horario=r, fijadas=[r.sesiones[0]]), opciones=list(opciones),
                    faltan_horas=2.5, fecha_critica=INICIO + timedelta(days=5))


def test_sin_decision_pendiente():
    """Si no hay decisión pendiente, leer devuelve None y descartar no falla."""
    con = base()
    e = nuevo_estudiante(con)
    assert leer_decision_pendiente(con, e) is None
    descartar_decision_pendiente(con, e)


def test_leer_devuelve_la_decision_y_no_la_borra():
    """La decisión vuelve igual a como se guardó (con su plan, horario y fijadas) y se puede leer las veces que haga falta."""
    con = base()
    e = nuevo_estudiante(con)
    d = decision_ejemplo()
    guardar_decision_pendiente(con, e, d)
    assert leer_decision_pendiente(con, e) == d
    assert leer_decision_pendiente(con, e) == d and contar(con, "decision_pendiente") == 1


def test_decision_nueva_reemplaza_a_la_anterior():
    """Si se guarda otra decisión antes de resolver la primera, la que espera es la última."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_decision_pendiente(con, e, decision_ejemplo())
    segunda = decision_ejemplo(("abrir_tiempo", "recortar"))
    guardar_decision_pendiente(con, e, segunda)
    assert contar(con, "decision_pendiente") == 1 and leer_decision_pendiente(con, e) == segunda


def test_descartar_borra_la_decision():
    """Cuando el estudiante cancela el cambio, la decisión se borra."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_decision_pendiente(con, e, decision_ejemplo())
    descartar_decision_pendiente(con, e)
    assert leer_decision_pendiente(con, e) is None and contar(con, "decision_pendiente") == 0


def test_decision_vencida_no_se_lee():
    """Una decisión de hace más de 30 minutos ya no cuenta como pendiente."""
    con = base()
    e = nuevo_estudiante(con)
    guardar_decision_pendiente(con, e, decision_ejemplo())
    hace_31_min = (datetime.now() - timedelta(minutes=31)).isoformat(timespec="seconds")
    with con:
        con.execute("UPDATE decision_pendiente SET creada_en = ?", (hace_31_min,))
    assert leer_decision_pendiente(con, e) is None


def test_decision_de_otro_estudiante_no_se_ve():
    """La decisión pendiente de un estudiante no la puede leer ni descartar otro."""
    con = base()
    e1, e2 = nuevo_estudiante(con), nuevo_estudiante(con)
    d = decision_ejemplo()
    guardar_decision_pendiente(con, e1, d)
    descartar_decision_pendiente(con, e2)
    assert leer_decision_pendiente(con, e2) is None
    assert leer_decision_pendiente(con, e1) == d


def test_decision_de_estudiante_inexistente():
    """No se puede guardar una decisión para un identificador que no está registrado."""
    con = base()
    falla_integridad(lambda: guardar_decision_pendiente(con, "no-existe", decision_ejemplo()), "FOREIGN KEY")


def test_decision_y_orden_pendiente_coexisten():
    """Tomar la orden que responde a la decisión no borra la decisión."""
    con = base()
    e = nuevo_estudiante(con)
    d = decision_ejemplo()
    guardar_decision_pendiente(con, e, d)
    guardar_orden_pendiente(con, e, ORDEN_MOVER)
    assert tomar_orden_pendiente(con, e) == ORDEN_MOVER
    assert leer_decision_pendiente(con, e) == d


def test_guardar_plan_borra_la_decision():
    """Al guardar un plan nuevo, la decisión pendiente se borra, porque hablaba del plan anterior."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    guardar_decision_pendiente(con, e, decision_ejemplo())
    guardar_plan(con, e, p, INICIO, r)
    assert leer_decision_pendiente(con, e) is None and contar(con, "decision_pendiente") == 0


def test_si_guardar_el_plan_falla_la_decision_sigue():
    """Si el guardado falla a mitad de camino, la transacción se deshace y la decisión sigue pendiente."""
    con = base()
    e = nuevo_estudiante(con)
    p, r = ejemplo()
    d = decision_ejemplo()
    guardar_decision_pendiente(con, e, d)
    roto = r.model_copy(update={"sesiones": r.sesiones + [r.sesiones[0]]})
    falla_integridad(lambda: guardar_plan(con, e, p, INICIO, roto), "UNIQUE")
    assert leer_decision_pendiente(con, e) == d


correr(test_sin_decision_pendiente, test_leer_devuelve_la_decision_y_no_la_borra, test_decision_nueva_reemplaza_a_la_anterior,
       test_descartar_borra_la_decision, test_decision_vencida_no_se_lee, test_decision_de_otro_estudiante_no_se_ve,
       test_decision_de_estudiante_inexistente, test_decision_y_orden_pendiente_coexisten, test_guardar_plan_borra_la_decision,
       test_si_guardar_el_plan_falla_la_decision_sigue)

ok test_sin_decision_pendiente
ok test_leer_devuelve_la_decision_y_no_la_borra
ok test_decision_nueva_reemplaza_a_la_anterior
ok test_descartar_borra_la_decision
ok test_decision_vencida_no_se_lee
ok test_decision_de_otro_estudiante_no_se_ve
ok test_decision_de_estudiante_inexistente
ok test_decision_y_orden_pendiente_coexisten
ok test_guardar_plan_borra_la_decision
ok test_si_guardar_el_plan_falla_la_decision_sigue


# PASO 5. UN EJEMPLO COMPLETO

Un estudiante pasa por las situaciones del apartado 1.1 con una base en un archivo de verdad (en una carpeta temporal), para comprobar que lo guardado sobrevive a cerrar la conexión. El intérprete y el MLP no participan: las órdenes y las predicciones van escritas a mano, porque ya se probaron en sus notebooks.

## 5.1 Primera visita: crear el plan

In [10]:
import pandas as pd


def ver(con: sqlite3.Connection, tabla: str) -> pd.DataFrame:
    """Muestra una tabla de la base sin la columna del identificador, que es larga y se repite en todas las filas."""
    return pd.read_sql(f"SELECT * FROM {tabla}", con).drop(columns=["estudiante_id"], errors="ignore")


ruta = str(Path(tempfile.mkdtemp()) / "studyflow.db")
con = conectar(ruta)
yo = nuevo_estudiante(con)  # el front guarda este identificador

# El intérprete entendió crear plan: la orden espera la confirmación del estudiante
guardar_orden_pendiente(con, yo, ORDEN_CREAR)
print("Órdenes pendientes:", contar(con, "orden_pendiente"))

# El estudiante confirma: se toma la orden, se arma el plan con su disponibilidad y se corre el flujo
orden = tomar_orden_pendiente(con, yo)
p = PlanEntrada(actividades=orden.actividades, disponibilidad=dias(0, 10), duracion_sesion=60)
print("Validador:", validar(p, INICIO).estado.value)
r = asignar(p, PREDS_EJEMPLO, INICIO)  # aquí iría el MLP
guardar_plan(con, yo, p, INICIO, r)
print("Órdenes pendientes:", contar(con, "orden_pendiente"), "| sesiones guardadas:", contar(con, "sesiones"))
con.close()  # el estudiante cierra la página

Órdenes pendientes: 1
Validador: factible
Órdenes pendientes: 0 | sesiones guardadas: 15


## 5.2 Vuelve otro día

Se abre una conexión nueva al mismo archivo, como hará la API en cada petición.

In [11]:
con = conectar(ruta)
g = cargar_plan(con, yo)
assert g.plan == p and g.horario == r
print(f"Plan recuperado: {len(g.plan.actividades)} actividades y {len(g.horario.sesiones)} sesiones")
display(ver(con, "planes"), ver(con, "actividades"), ver(con, "resumen"))

Plan recuperado: 2 actividades y 15 sesiones


,fecha_inicio,duracion_sesion,max_horas_dia,actualizado_en
0,2026-10-25,60,None,2026-11-09T00:25:58


,posicion,nombre,prioridad,dificultad,fecha_limite,horas_estimadas
0,0,Parcial de IA,alta,alta,2026-10-31,8.0
1,1,Taller,media,media,2026-10-30,4.0


,posicion,nombre,prioridad,horas_solicitadas,horas_asignadas,horas_faltantes,estado
0,0,Parcial de IA,0.87,8.0,8.0,0.0,asignada
1,1,Taller,0.55,4.0,4.0,0.0,asignada


## 5.3 Mañana no puedo estudiar

El estudiante bloquea el lunes 26 (n=1). La orden espera confirmación, el estudiante confirma y se replanifica a partir del plan guardado.

In [12]:
guardar_orden_pendiente(con, yo, Orden(accion="bloquear_dia", dia=f(1)))
orden = tomar_orden_pendiente(con, yo)

g = cargar_plan(con, yo)
nuevo = g.plan.model_copy(update={"disponibilidad": [d for d in g.plan.disponibilidad if d.dia != orden.dia]})
print("Validador:", validar(nuevo, g.fecha_inicio, "replanificacion").estado.value)
r2 = asignar(nuevo, PREDS_EJEMPLO, g.fecha_inicio)
guardar_plan(con, yo, nuevo, g.fecha_inicio, r2)

print("Sesiones el día bloqueado:", con.execute("SELECT COUNT(*) FROM sesiones WHERE dia = ?", (orden.dia.isoformat(),)).fetchone()[0])
display(ver(con, "resumen"))
ver(con, "sesiones")

Validador: factible
Sesiones el día bloqueado: 0


,posicion,nombre,prioridad,horas_solicitadas,horas_asignadas,horas_faltantes,estado
0,0,Parcial de IA,0.87,8.0,8.0,0.0,asignada
1,1,Taller,0.55,4.0,4.0,0.0,asignada


,dia,inicio,fin,actividad,fijada
0,2026-10-25,18:00:00,19:00:00,Parcial de IA,0
1,2026-10-25,19:00:00,20:00:00,Parcial de IA,0
2,2026-10-25,20:00:00,21:00:00,Taller,0
3,2026-10-27,18:00:00,19:00:00,Parcial de IA,0
4,2026-10-27,19:00:00,19:30:00,Parcial de IA,0
5,2026-10-27,19:30:00,20:00:00,Taller,0
6,2026-10-28,18:00:00,19:00:00,Parcial de IA,0
7,2026-10-28,19:00:00,19:30:00,Parcial de IA,0
8,2026-10-28,19:30:00,20:30:00,Taller,0
9,2026-10-28,20:30:00,21:00:00,Taller,0


**Observaciones**

1. La orden de crear el plan esperó en la base hasta que el estudiante confirmó, y al tomarla se borró (`Órdenes pendientes: 1` y luego `0`).
2. Después de cerrar la conexión y abrir otra, el plan y el horario vuelven idénticos a los guardados (el `assert` lo comprueba): 2 actividades y 15 sesiones. La prioridad del MLP (0.87 y 0.55) queda en `resumen`, así que el front la muestra sin volver a predecir.
3. Al bloquear el lunes 26, el plan se recuperó de la base, se le quitó ese día y se replanificó. No quedan sesiones ese día y las dos actividades siguen `asignada`, con sus 12 h repartidas en los demás días. `guardar_plan` reemplazó el horario anterior completo.
4. Aquí se reutilizaron las mismas predicciones para replanificar; en la API el MLP vuelve a predecir con el plan nuevo.
5. En la base, fechas y horas se ven como texto ISO; al cargar, Pydantic las convierte otra vez en `date` y `time`.

## 5.4 Mueve una sesión y queda fijada

Aquí participa la capa de órdenes, con un MLP de mentira (`PredictorEjemplo`) que devuelve las predicciones escritas a mano, porque el MLP ya se probó en su notebook. Un estudiante nuevo crea su plan y mueve la sesión del Taller del lunes 26 al martes 27 a las 19:00. La sesión movida queda **fijada**: `guardar_plan` la guarda marcada y `cargar_plan` la devuelve en `fijadas`.

`vigente` arma lo que la capa de órdenes recibe a partir de lo que devuelve `cargar_plan`; es lo mismo que hará la API.

In [13]:
class PredictorEjemplo:
    """MLP de mentira para el ejemplo: devuelve las predicciones escritas a mano de cada actividad."""

    TABLA = {"Parcial de IA": PREDS_EJEMPLO[0], "Taller": PREDS_EJEMPLO[1]}

    def predecir(self, plan: PlanEntrada, fecha_inicio: date) -> list[Prediccion]:
        """Una predicción por actividad del plan."""
        return [self.TABLA[a.nombre] for a in plan.actividades]


def vigente(g: PlanGuardado) -> PlanVigente:
    """Lo que la capa de órdenes recibe: el plan, el horario y las sesiones fijadas guardados."""
    return PlanVigente(plan=g.plan, horario=g.horario, fijadas=g.fijadas)


mlp = PredictorEjemplo()
ahora = datetime.combine(INICIO, time(10, 0))  # domingo 25 de octubre, 10:00

ruta2 = str(Path(tempfile.mkdtemp()) / "studyflow.db")
con = conectar(ruta2)
ana = nuevo_estudiante(con)

# Crea su plan con la capa de órdenes y lo guarda
p = PlanEntrada(**plan([act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)], dias(0, 10)))
v = crear_plan(p, ahora, mlp).vigente
guardar_plan(con, ana, v.plan, INICIO, v.horario, v.fijadas)
print("Fijadas al crear el plan:", len(cargar_plan(con, ana).fijadas))

# Mueve la sesión del Taller del lunes 26 al miércoles 27 a las 19:00
g = cargar_plan(con, ana)
r = aplicar(vigente(g), Orden(accion="mover_sesion", actividad="Taller", dia_origen=f(1), dia_destino=f(2), hora_inicio="19:00"), ahora, mlp)
guardar_plan(con, ana, r.vigente.plan, g.fecha_inicio, r.vigente.horario, r.vigente.fijadas)

g = cargar_plan(con, ana)
assert r.tipo == "aplicada" and g.fijadas == r.vigente.fijadas and len(g.fijadas) == 1
print("Resultado:", r.tipo, "| fijadas guardadas:", len(g.fijadas))
ver(con, "sesiones").query("fijada == 1")

Fijadas al crear el plan: 0
Resultado: aplicada | fijadas guardadas: 1


,dia,inicio,fin,actividad,fijada
7,2026-10-27,19:00:00,19:30:00,Taller,1


## 5.5 Bloquea tres días y no cabe

El estudiante bloquea del miércoles 28 al viernes 30 (n=3 a n=5). Lo que le queda no alcanza para las 12 h, así que la capa de órdenes no cambia el plan: devuelve una **decisión**. La API la guarda y el plan guardado sigue siendo el mismo. Luego el estudiante cierra la página.

In [14]:
g = cargar_plan(con, ana)
r = aplicar(vigente(g), Orden(accion="bloquear_dia", dia=f(3), dia_fin=f(5)), ahora, mlp)
print(r.tipo, "|", r.mensaje)
guardar_decision_pendiente(con, ana, r.decision)

assert r.tipo == "decision" and cargar_plan(con, ana) == g  # el plan guardado no cambió
print("Decisiones pendientes:", contar(con, "decision_pendiente"), "| faltan", r.decision.faltan_horas, "h")
con.close()  # el estudiante cierra la página

decision | Con este cambio no alcanza el tiempo: faltan 3 h antes del sábado 31 de octubre. Puedes abrir más tiempo (más horas otros días o un máximo diario mayor) o recortar horas (dime qué actividades quieres mantener intactas).
Decisiones pendientes: 1 | faltan 3.0 h


## 5.6 Vuelve y abre tiempo

Con una conexión nueva, la API lee la decisión (sin borrarla) y la pasa a la capa de órdenes junto con la orden del estudiante: abrir el miércoles 28 de 18:00 a 21:00. Al guardar el plan nuevo se borra la decisión, y la sesión fijada del paso 5.4 sigue en su lugar.

In [15]:
con = conectar(ruta2)
d = leer_decision_pendiente(con, ana)
print("Opciones:", d.opciones)
assert leer_decision_pendiente(con, ana) == d  # se lee sin borrarla

g = cargar_plan(con, ana)
fijada = g.fijadas[0]
r = aplicar(vigente(g), Orden(accion="abrir_tiempo", dia=f(3), hora_inicio="18:00", hora_fin="21:00"), ahora, mlp, d)
guardar_plan(con, ana, r.vigente.plan, g.fecha_inicio, r.vigente.horario, r.vigente.fijadas)

g = cargar_plan(con, ana)
assert r.tipo == "aplicada" and leer_decision_pendiente(con, ana) is None
assert g.fijadas == [fijada] and all(a.estado.value == "asignada" for a in g.horario.actividades)
print("Resultado:", r.tipo, "| decisiones pendientes:", contar(con, "decision_pendiente"), "| fijadas:", len(g.fijadas))
display(ver(con, "resumen"))
con.close()

Opciones: ['abrir_tiempo', 'recortar']
Resultado: aplicada | decisiones pendientes: 0 | fijadas: 1


,posicion,nombre,prioridad,horas_solicitadas,horas_asignadas,horas_faltantes,estado
0,0,Parcial de IA,0.87,8.0,8.0,0.0,asignada
1,1,Taller,0.55,4.0,4.0,0.0,asignada


**Observaciones**

1. Al crear el plan no hay fijadas. Al mover la sesión del Taller del lunes 26 al martes 27 queda una fijada: en la base es la fila de `sesiones` con `fijada = 1`, y el horario guardado sigue siendo el completo.
2. Bloquear del miércoles 28 al viernes 30 no cabe (faltan 3 h antes del sábado 31 de octubre), así que `aplicar` devolvió una decisión con las opciones abrir tiempo y recortar. No incluye cancelar, porque un bloqueo ya ocurrió. La API guardó la decisión y el plan guardado quedó idéntico: el `assert` compara el `PlanGuardado` completo.
3. Con una conexión nueva la decisión sigue ahí, y leerla dos veces da lo mismo. Al abrir el miércoles 28 de 18:00 a 21:00 las 12 h caben; `guardar_plan` reemplaza el plan y la decisión desaparece en la misma transacción (0 pendientes).
4. La fijada del Taller sobrevivió al cambio: la capa de órdenes la respetó al replanificar y `guardar_plan` la volvió a guardar marcada.

# PASO 6. PRUEBA CON PLANES ALEATORIOS

Los mismos planes al azar de la capa de asignación (1 a 5 actividades, franjas y topes variados), ahora con días bloqueados de vez en cuando. Cada plan que el validador no declara inviable se asigna con predicciones aleatorias, se guarda para uno de 5 estudiantes elegido al azar (reemplazando lo que tenía), con cerca del 20 % de sus sesiones fijadas al azar, y se vuelve a cargar. Al final se comprueba que cada estudiante recupera su último plan con sus fijadas y que no quedan filas de planes anteriores.

In [16]:
import random

rng = random.Random(7)
rng_fijadas = random.Random(11)  # aparte, para no cambiar los planes al azar
NIV = ["baja", "media", "alta"]
FRANJAS = [(("18:00", "21:00"),), (("08:00", "10:00"), ("14:00", "16:30")), (("19:00", "20:30"),), (("16:00", "17:50"), ("19:00", "21:00"))]


def plan_aleatorio() -> PlanEntrada:
    """Plan al azar con 16 días, de los cuales cerca del 15 % quedan bloqueados."""
    actividades = [act(f"A{i}", rng.randint(3, 14), rng.choice([1, 1.5, 2, 3, 4, 5, 6]), rng.choice(NIV), rng.choice(NIV))
                   for i in range(rng.randint(1, 5))]
    disponibilidad = [{"dia": f(n), "franjas": franjas(*rng.choice(FRANJAS)) if rng.random() < 0.85 else []} for n in range(16)]
    return PlanEntrada(**plan(actividades, disponibilidad, rng.choice([60, 90, 120]), rng.choice([None, 3, 4])))


def prediccion_aleatoria() -> Prediccion:
    """Prioridad y perfil al azar, sin el MLP."""
    w = [rng.random() for _ in range(3)]
    return Prediccion(prioridad=rng.random(), perfil=[x / sum(w) for x in w])


con = base()
estudiantes = [nuevo_estudiante(con) for _ in range(5)]
ultimo, guardados = {}, 0
for _ in range(1000):
    p = plan_aleatorio()
    if validar(p, INICIO).estado == Estado.INVIABLE:
        continue
    r = asignar(p, [prediccion_aleatoria() for _ in p.actividades], INICIO)
    fijadas = [s for s in r.sesiones if rng_fijadas.random() < 0.2]
    e = rng.choice(estudiantes)
    guardar_plan(con, e, p, INICIO, r, fijadas)
    g = cargar_plan(con, e)
    assert g.plan == normalizar(p) and g.horario == r and g.fijadas == fijadas
    ultimo[e], guardados = (p, r, fijadas), guardados + 1

for e, (p, r, fijadas) in ultimo.items():
    g = cargar_plan(con, e)
    assert g.plan == normalizar(p) and g.horario == r and g.fijadas == fijadas
assert contar(con, "planes") == len(ultimo)
assert contar(con, "actividades") == sum(len(p.actividades) for p, _, _ in ultimo.values())
assert contar(con, "sesiones") == sum(len(r.sesiones) for _, r, _ in ultimo.values())
assert con.execute("SELECT COUNT(*) FROM sesiones WHERE fijada = 1").fetchone()[0] == sum(len(fj) for _, _, fj in ultimo.values())

print(f"{guardados} planes guardados y recuperados sin diferencias, repartidos entre {len(ultimo)} estudiantes")
print(f"Filas al final: {contar(con, 'actividades')} actividades, {contar(con, 'franjas')} franjas, {contar(con, 'sesiones')} sesiones "
      f"({sum(len(fj) for _, _, fj in ultimo.values())} fijadas)")

874 planes guardados y recuperados sin diferencias, repartidos entre 5 estudiantes
Filas al final: 17 actividades, 94 franjas, 81 sesiones (14 fijadas)


**Observaciones**

1. De los 1000 planes al azar (semilla 7), 874 no fueron inviables. Los 874 se guardaron y se recuperaron sin ninguna diferencia, incluidos los que tenían días bloqueados, que vuelven sin esos días como se decidió en 1.4.
2. Cada guardado reemplazó el plan anterior del estudiante: al final la base tiene solo los 5 planes vigentes (17 actividades, 94 franjas y 81 sesiones, 14 de ellas fijadas) y ninguna fila de los otros 869.
3. Las prioridades del MLP, con todos sus decimales, vuelven exactas: SQLite guarda `REAL` con la misma precisión que un `float` de Python.
4. Las fijadas (cerca del 20 % de las sesiones de cada horario, elegidas al azar) vuelven siempre iguales, y cada guardado reemplaza las del plan anterior.

# PASO 7. MIGRACIÓN A ARCHIVOS `.py`

| Celdas del notebook | Archivo | Imports que se agregan |
|---|---|---|
| Pasos 2, 3 y 4, celdas de `persistencia.py` | `studyflow/persistencia.py` (una sola, en ese orden) | `from studyflow.models import PlanEntrada`, `from studyflow.asignacion import ResultadoAsignacion, Sesion`, `from studyflow.interprete import Orden` y `from studyflow.ordenes import Decision` |
| Pasos 2.1, 3.2, 4.1, 4.2 y 6 | `tests/test_persistencia.py` | `base` pasa a ser un fixture de pytest; `INICIO`, `act`, `dias`, `plan` y `franjas` ya están en `conftest.py`; `contar`, `normalizar`, `PREDS_EJEMPLO` y `falla_integridad` van al archivo de pruebas |

Las funciones `test_...` pasan sin cambios: pytest las encuentra por su nombre y `correr` ya no hace falta.

**Antes de commitear:** agregar `*.db` al `.gitignore` (hoy solo ignora `db.sqlite3`), para que una base con datos de estudiantes nunca suba al repositorio.

## Puntos para tener en cuenta en la API (FastAPI)

* **Una conexión por petición.** Una conexión de `sqlite3` no se comparte entre hilos, y FastAPI atiende peticiones en paralelo: cada ruta abre la suya con `conectar` y la cierra al terminar.
* **Ruta de la base:** conviene leerla del `.env` (por ejemplo `STUDYFLOW_DB`), igual que la clave del LLM, en vez de dejar fija `studyflow.db`.
* **De la base a la capa de órdenes y de vuelta.** `PlanVigente(plan=g.plan, horario=g.horario, fijadas=g.fijadas)` arma lo que recibe `aplicar`, y `guardar_plan(con, e, v.plan, fecha_inicio, v.horario, v.fijadas)` guarda el resultado, con la `fecha_inicio` que corresponda.
* **Decisión pendiente:** si `leer_decision_pendiente` devuelve `None` (no había o ya venció), la API trata el mensaje como si no hubiera decisión abierta y el intérprete no acepta `responder_decision`. Después de cada resultado `decision` (o `pregunta`) la API llama a `guardar_decision_pendiente`; si la respuesta fue cancelar, a `descartar_decision_pendiente`. No hace falta borrarla al aplicar un cambio: `guardar_plan` ya lo hace.
* **Identificador del estudiante:** el front lo guarda (por ejemplo en `localStorage`) y lo envía en un encabezado, no en la URL. Si llega uno que no existe (`existe_estudiante` da `False`), la API responde con error y el front pide uno nuevo.
* **Plan inviable:** `guardar_plan` solo se llama si el validador no declara el plan inviable. Así el estudiante conserva su plan anterior y se le explica por qué no se pudo.
* **Datos sin login:** nada borra a los estudiantes que no vuelven. Si el proyecto se publica, convendría limpiar los inactivos (por ejemplo, después de 30 días sin actualizar).